In [1]:
import pandas as pd
import numpy as np
from scipy import stats
from sklearn.metrics import confusion_matrix
from sklearn.naive_bayes import GaussianNB

df = pd.read_csv('KO_weekly_return_volatility.csv')
daily_data = pd.read_csv('KO_weekly_return_detailed.csv')

train = df[df['Year'].isin([2021, 2022, 2023])].reset_index(drop=True)
test = df[df['Year'].isin([2024, 2025])].reset_index(drop=True)
daily_test = daily_data[daily_data['Year'].isin([2024, 2025])].reset_index(drop=True)

X_train = train[['mean_return', 'volatility']].values
Y_train = train['decision'].values
X_test = test[['mean_return', 'volatility']].values
Y_test = test['decision'].values

print(X_train.shape)

classes = ['YES', 'NO']

def predict_student_t(X_train, Y_train, X_test, df_fixed):
    priors = {cls: np.mean(Y_train == cls) for cls in classes}
    params = {}
    for cls in classes:
        X_cls = X_train[Y_train == cls]
        params[cls] = []
        for j in range(X_train.shape[1]):
            _, loc, scale = stats.t.fit(X_cls[:, j])
            params[cls].append((df_fixed, loc, scale))
    preds = []
    for x in X_test:
        scores = {}
        for cls in classes:
            log_prob = np.log(priors[cls])
            for j, (df_t, loc, scale) in enumerate(params[cls]):
                log_prob += np.log(stats.t.pdf(x[j], df=df_t, loc=loc, scale=scale) + 1e-300)
            scores[cls] = log_prob
        preds.append(max(scores, key=scores.get))
    return np.array(preds)

(156, 2)


## Q1 : Implement Student-t Naive Bayes (df = 0.5, 1, 5) and compute accuracy for testing period

In [2]:
results = {}
for df_val in [0.5, 1, 5]:
    preds = predict_student_t(X_train, Y_train, X_test, df_val)
    acc = round(np.mean(preds == Y_test) * 100, 2)
    results[df_val] = (acc, preds)
    print(f'df={df_val}, accuracy={acc}%')

df=0.5, accuracy=68.87%
df=1, accuracy=68.87%
df=5, accuracy=68.87%


## Q2 : Confusion matrices for testing period


In [3]:
for df_val in [0.5, 1, 5]:
    preds = results[df_val][1]
    cm = confusion_matrix(Y_test, preds, labels=['YES', 'NO'])
    print(f'\ndf={df_val}:')
    print(f'  Predicted YES  Predicted NO')
    print(f'Actual YES  {cm[0][0]}           {cm[0][1]}')
    print(f'Actual NO   {cm[1][0]}           {cm[1][1]}')


df=0.5:
  Predicted YES  Predicted NO
Actual YES  2           31
Actual NO   2           71

df=1:
  Predicted YES  Predicted NO
Actual YES  0           33
Actual NO   0           73

df=5:
  Predicted YES  Predicted NO
Actual YES  0           33
Actual NO   0           73


## Q3 : True positive rate and true negative rate for testing period


In [4]:
for df_val in [0.5, 1, 5]:
    preds = results[df_val][1]
    cm = confusion_matrix(Y_test, preds, labels=['YES', 'NO'])
    TP, FN, FP, TN = cm[0][0], cm[0][1], cm[1][0], cm[1][1]
    print(f'df={df_val}: TPR={round(TP/(TP+FN)*100,2)}%, TNR={round(TN/(TN+FP)*100,2)}%')

df=0.5: TPR=6.06%, TNR=97.26%
df=1: TPR=0.0%, TNR=100.0%
df=5: TPR=0.0%, TNR=100.0%


## Q4 : Best value of df, is it better than normal Gaussian NB?


In [5]:
gnb = GaussianNB()
gnb.fit(X_train, Y_train)
gnb_acc = round(gnb.score(X_test, Y_test) * 100, 2)

best_df = max(results, key=lambda x: results[x][0])
print(f'Best df={best_df} with accuracy={results[best_df][0]}%')
print(f'Normal Gaussian NB accuracy={gnb_acc}%')
print(f'{"Student-t is better" if results[best_df][0] > gnb_acc else "Normal NB is better or equal"}')

Best df=0.5 with accuracy=68.87%
Normal Gaussian NB accuracy=69.81%
Normal NB is better or equal


## Q5 : Trading strategy using best df vs buy-and-hold


In [6]:
best_preds = results[best_df][1]
test['t_prediction'] = best_preds

initial_balance = 100
t_balance = initial_balance
position = None

for week_idx in range(len(test)):
    current_week = test.iloc[week_idx]
    year = current_week['Year']
    week_num = current_week['Week_Number']
    week_data = daily_test[(daily_test['Year'] == year) & (daily_test['Week_Number'] == week_num)]

    if len(week_data) == 0:
        continue

    if position == 'invested':
        for idx, row in week_data.iterrows():
            if pd.notna(row['Return']):
                t_balance = t_balance * (1 + row['Return'] / 100)

    if week_idx < len(test) - 1:
        next_prediction = test.iloc[week_idx + 1]['t_prediction']
        position = 'invested' if next_prediction == 'YES' else None

bh_balance = initial_balance
for idx, row in daily_test.iterrows():
    if pd.notna(row['Return']):
        bh_balance = bh_balance * (1 + row['Return'] / 100)

print(f'Student-t (df={best_df}) Strategy Final Balance: ${t_balance:.2f}')
print(f'Buy and Hold Final Balance: ${bh_balance:.2f}')
winner = f'Student-t (df={best_df})' if t_balance > bh_balance else 'Buy-and-Hold'
print(f'{winner} results in a larger amount at the end of the testing period')

Student-t (df=0.5) Strategy Final Balance: $103.22
Buy and Hold Final Balance: $126.14
Buy-and-Hold results in a larger amount at the end of the testing period


## Q6 : NB using Laplace (non-symmetric) distribution, are results better?


In [7]:
def predict_laplace_nb(X_train, Y_train, X_test):
    priors = {cls: np.mean(Y_train == cls) for cls in classes}
    params = {}
    for cls in classes:
        X_cls = X_train[Y_train == cls]
        params[cls] = []
        for j in range(X_train.shape[1]):
            loc, scale = stats.laplace.fit(X_cls[:, j])
            params[cls].append((loc, scale))
    preds = []
    for x in X_test:
        scores = {}
        for cls in classes:
            log_prob = np.log(priors[cls])
            for j, (loc, scale) in enumerate(params[cls]):
                log_prob += np.log(stats.laplace.pdf(x[j], loc=loc, scale=scale) + 1e-300)
            scores[cls] = log_prob
        preds.append(max(scores, key=scores.get))
    return np.array(preds)

laplace_preds = predict_laplace_nb(X_train, Y_train, X_test)
laplace_acc = round(np.mean(laplace_preds == Y_test) * 100, 2)
print(f'Laplace NB accuracy={laplace_acc}%')
print(f'{"Laplace is better" if laplace_acc > gnb_acc else "Normal NB is better or equal"}')

Laplace NB accuracy=68.87%
Normal NB is better or equal
